# EP07. Context Compression
## 긴 문서를 토큰 낭비 없이 쑤셔 넣는 압축 기술

---

### 학습 목표

1. **컨텍스트 압축 기법 이해** — 요약 기반 vs 추출 기반 압축의 차이와 트레이드오프 파악
2. **LangChain ContextualCompressionRetriever 구현** — LLMChainFilter, LLMChainExtractor, EmbeddingsFilter, Pipeline 구성
3. **압축률 / 성능 측정** — tiktoken으로 토큰 수 계산, QA 정확도 자동 평가
4. **Langfuse 비용 추적** — 압축 전후 토큰 사용량 및 비용을 Langfuse 대시보드에서 모니터링

---

### AI 가이드 안내

> 이 노트북은 Claude Code 또는 ChatGPT와 함께 학습하도록 설계되어 있습니다.  
> 각 셀 하단의 `# TODO` 주석이 있는 부분은 직접 구현해 보세요.  
> 막히면 AI에게 "이 셀의 TODO를 구현해줘" 라고 질문하면 됩니다.

**권장 질문 예시**
- "LLMChainFilter와 LLMChainExtractor의 차이를 코드로 설명해줘"
- "DocumentCompressorPipeline을 두 압축기로 구성하는 방법을 알려줘"
- "tiktoken으로 한국어 토큰 수를 정확하게 세는 방법은?"

---
## 섹션 1. 환경 설정

필요한 패키지를 설치합니다. `uv`를 사용하면 pip보다 훨씬 빠릅니다.

In [ ]:
# uv pip install (권장)
# !uv pip install langchain langchain-anthropic langchain-openai langchain-community \
#     langfuse chromadb sentence-transformers tiktoken python-dotenv llmlingua

# pip 사용 시
!pip install -q langchain langchain-anthropic langchain-openai langchain-community \
    langfuse chromadb sentence-transformers tiktoken python-dotenv

---
## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# .env 파일에서 API 키 로드
from dotenv import load_dotenv
load_dotenv()

# 필수 환경 변수 확인
required_keys = ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"]
for key in required_keys:
    val = os.getenv(key)
    status = "OK" if val else "MISSING"
    print(f"{key}: {status}")

In [ ]:
# LangChain 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import (
    LLMChainFilter,
    LLMChainExtractor,
    EmbeddingsFilter,
    DocumentCompressorPipeline,
)
from langchain.schema import Document

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("라이브러리 로드 완료")

---
## 섹션 3. 긴 문서 샘플 준비

Python 라이브러리 문서 스타일의 긴 텍스트 5개를 인라인으로 정의합니다.  
합쳐서 약 3,000단어 분량입니다.

In [ ]:
# 문서 1: LangChain 개요
DOC_LANGCHAIN = """
LangChain is a framework for developing applications powered by large language models (LLMs).
It was created by Harrison Chase and first released in October 2022. The framework provides
a standardized interface for working with LLMs and related tools, enabling developers to
build complex AI applications with minimal boilerplate code.

The core philosophy of LangChain is composability. Rather than providing a monolithic
solution, it offers modular components that can be combined in various ways. The main
abstractions include: Models (LLMs and Chat Models), Prompts (templates and selectors),
Chains (sequences of calls), Agents (LLMs that decide actions), Memory (state persistence),
and Retrievers (document fetchers).

LangChain Expression Language (LCEL) is the recommended way to compose chains. It uses
Python's pipe operator (|) to connect components in a readable, functional style. LCEL
supports streaming, async execution, parallel processing, and provides built-in retry
and fallback mechanisms.

The retrieval augmented generation (RAG) pattern is one of LangChain's most popular use
cases. In a basic RAG pipeline, documents are split into chunks, embedded into a vector
space, stored in a vector database, and retrieved based on semantic similarity to the
user's query. The retrieved context is then passed to an LLM to generate a grounded response.

LangChain supports dozens of vector stores including Chroma, Pinecone, Weaviate, FAISS,
and Qdrant. It also integrates with many embedding providers such as OpenAI, Anthropic,
Cohere, HuggingFace, and local models via Ollama.

Context compression is a technique in LangChain that reduces the amount of text passed
to the LLM after retrieval. The ContextualCompressionRetriever wraps a base retriever
and applies a compressor to the retrieved documents. Available compressors include
LLMChainFilter (removes irrelevant chunks), LLMChainExtractor (extracts relevant sentences),
and EmbeddingsFilter (filters by embedding similarity). Multiple compressors can be
combined using DocumentCompressorPipeline.

LangSmith is LangChain's observability platform. It provides detailed tracing of every
LLM call, token usage statistics, latency measurements, and tools for evaluating
chain quality. Langfuse is a popular open-source alternative to LangSmith that can
be integrated via a simple callback handler.

The LangChain ecosystem consists of several packages: langchain-core (base abstractions),
langchain (main package with chains and agents), langchain-community (third-party integrations),
and model-specific packages like langchain-openai and langchain-anthropic.
"""

# 문서 2: Retrieval Augmented Generation 심화
DOC_RAG = """
Retrieval Augmented Generation (RAG) is an AI architecture that enhances the capabilities
of large language models by incorporating an external knowledge retrieval step before
generation. Introduced by Lewis et al. in 2020, RAG addresses the fundamental limitation
of LLMs: their knowledge is frozen at training time.

The RAG pipeline consists of two phases: indexing and retrieval-generation. During indexing,
source documents are loaded, split into smaller chunks (typically 500-1500 tokens), embedded
into dense vector representations, and stored in a vector database. The chunk size is a
critical hyperparameter: larger chunks retain more context but reduce retrieval precision,
while smaller chunks improve precision but may lose coherence.

During the retrieval-generation phase, the user's query is embedded using the same model
as the documents. The vector database performs approximate nearest neighbor (ANN) search
to find the top-k most similar chunks. These chunks are concatenated into a context
window and passed along with the query to the LLM for generation.

Advanced RAG techniques address the shortcomings of naive RAG. Query transformation
techniques include HyDE (Hypothetical Document Embeddings), multi-query retrieval, and
step-back prompting. Post-retrieval techniques include re-ranking with cross-encoders,
context compression, and reciprocal rank fusion for combining multiple retrievers.

Context compression in RAG specifically refers to reducing the token count of retrieved
documents while preserving the information needed to answer the query. This is important
because retrieved chunks often contain large amounts of irrelevant text. A typical
compression pipeline might first filter chunks by embedding similarity, then use an LLM
to extract only the sentences directly relevant to the query.

Evaluation of RAG systems typically uses metrics such as faithfulness (does the answer
match the retrieved context?), answer relevancy (does the answer address the question?),
and context precision/recall (are the retrieved chunks relevant and complete?). RAGAS is
a popular framework for automated RAG evaluation.

The choice between RAG and long-context LLMs is increasingly important as models like
Claude 3 (200K context) and Gemini 1.5 (1M context) expand their windows. RAG remains
preferable when the knowledge base exceeds the context window, when fresh data is needed,
or when cost efficiency is paramount. Long context is better when holistic understanding
of a single document is required.
"""

# 문서 3: Vector Databases 비교
DOC_VECTORDB = """
Vector databases are purpose-built systems for storing, indexing, and querying high-dimensional
vector embeddings. They have become essential infrastructure for AI applications, particularly
those using RAG or semantic search. The key operations are insert (store embedding + metadata),
query (find top-k nearest neighbors), update, and delete.

Chroma is an open-source, embedded vector database designed for developer ease of use.
It runs in-process (no server required), supports SQLite persistence, and has a simple
Python API. Chroma uses HNSW (Hierarchical Navigable Small World) for approximate nearest
neighbor search. It is ideal for prototyping and small-to-medium datasets up to a few
million vectors.

Pinecone is a managed cloud vector database offering high scalability and low latency.
It supports serverless and pod-based deployment, metadata filtering, and namespaces for
data isolation. Pinecone charges based on storage and query volume, making it cost-effective
at scale but potentially expensive for development.

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search,
not a full database. It supports various index types including Flat (exact search),
IVF (inverted file index), and HNSW. FAISS is extremely fast but lacks metadata filtering
and persistence (must be manually serialized).

Weaviate is an open-source vector database with built-in vectorization modules,
GraphQL API, and support for both vector and keyword search (hybrid search). It can be
self-hosted or used via Weaviate Cloud. Weaviate's schema system allows complex data
modeling with cross-references between objects.

Qdrant is a vector database written in Rust, known for high performance and rich
filtering capabilities. It supports payload (metadata) indexing, scalar and binary
quantization for memory reduction, and distributed deployment. Qdrant's sparse vector
support enables hybrid dense-sparse search.

When selecting a vector database, key considerations include: scale (number of vectors),
query latency requirements, metadata filtering needs, hosting preferences (embedded vs
server vs managed), cost model, and existing infrastructure. For LangChain integration,
all major vector databases are supported via langchain-community.
"""

# 문서 4: Token Optimization 기법
DOC_TOKENS = """
Token optimization is the practice of reducing the number of tokens consumed in LLM
interactions while maintaining output quality. Since LLM APIs charge per token and have
finite context windows, token optimization directly impacts cost and capability.

Tokenization is the process of converting text into numerical tokens that LLMs can process.
Different tokenizers handle text differently: OpenAI's tiktoken uses byte-pair encoding (BPE),
resulting in approximately 0.75 tokens per word for English text. Non-English text, code,
and special characters typically use more tokens per character.

Prompt optimization techniques include: removing redundant instructions, using concise
language, avoiding repeated context, using system prompts efficiently, and leveraging
model capabilities to infer obvious requirements. A well-optimized prompt can achieve
the same result with 50-70% fewer tokens.

Context window management strategies: For long conversations, older messages can be
summarized rather than truncated. Rolling summaries compress conversation history while
preserving key information. Entity memory systems extract and store important entities
mentioned in conversation for compact recall.

Document compression for RAG: Before passing retrieved documents to the LLM, various
compression techniques can reduce token usage. Extractive compression selects the most
relevant sentences. Abstractive compression generates a summary. Hybrid approaches
combine both methods.

LLMlingua is a Microsoft Research project that uses a small language model (e.g., Phi-2)
to score token importance using perplexity. Tokens with low perplexity (easily predictable)
are removed, achieving compression ratios of 2x-20x with minimal quality loss. LLMlingua-2
improves on the original with better multilingual support and a dedicated compression model.

Caching strategies: Prompt caching (supported by Anthropic and OpenAI) can cache the
system prompt and repeated context, charging only for cache read tokens (much cheaper
than full input tokens). KV cache refers to the internal key-value cache in transformer
models that speeds up generation for cached prefixes.

Structured output optimization: JSON mode and function calling reduce output tokens
by enforcing compact formats. Streaming responses allow early termination when sufficient
information is obtained, avoiding unnecessary generation.
"""

# 문서 5: Langfuse Observability
DOC_LANGFUSE = """
Langfuse is an open-source observability and analytics platform for LLM applications.
It was founded in 2023 and provides detailed tracing, token usage analytics, cost tracking,
user feedback collection, and prompt management. Langfuse can be self-hosted or used
via Langfuse Cloud.

The core data model in Langfuse consists of Traces, Observations, and Scores. A Trace
represents a single end-to-end request (e.g., a RAG query). Observations are nested
spans within a trace, representing individual LLM calls, retrievals, or other operations.
Scores are custom numeric evaluations attached to traces.

LangChain integration is accomplished via the CallbackHandler from langfuse.langchain.
When passed as a callback to any LangChain component, it automatically captures all
LLM calls, including prompts, completions, token counts, and latencies. No code changes
to the LangChain application are required beyond adding the handler.

Token cost tracking in Langfuse uses configurable price models. For each model, input
and output token prices are defined. Langfuse automatically calculates cost based on
observed token usage. Custom price models can be added for self-hosted or fine-tuned
models. Cost is displayed per trace, per session, and aggregated over time periods.

The Langfuse Python SDK v4 (langfuse >= 3.0) introduces significant changes from v2.
The CallbackHandler import path changed to from langfuse.langchain import CallbackHandler.
The SDK supports both synchronous and asynchronous usage patterns.

Prompt management in Langfuse allows teams to version, test, and deploy prompts without
code deployments. Prompts stored in Langfuse can be fetched at runtime and used in
LangChain chains. This enables A/B testing prompts and rolling back problematic changes.

Evaluations in Langfuse can be performed manually (human labelers scoring outputs)
or automatically (LLM-as-judge, model-based scoring). Custom evaluation functions can
be registered and triggered after each trace. Evaluation results are linked to traces
for root-cause analysis.

Dataset management allows teams to create benchmark datasets, run experiments, and
compare results across model versions or prompt variations. Each experiment run produces
a summary with aggregate metrics, making it easy to identify regressions or improvements.
"""

# 모든 문서를 Document 객체로 변환
raw_docs = [
    Document(page_content=DOC_LANGCHAIN, metadata={"source": "langchain_overview", "topic": "LangChain"}),
    Document(page_content=DOC_RAG, metadata={"source": "rag_deep_dive", "topic": "RAG"}),
    Document(page_content=DOC_VECTORDB, metadata={"source": "vectordb_comparison", "topic": "VectorDB"}),
    Document(page_content=DOC_TOKENS, metadata={"source": "token_optimization", "topic": "Tokens"}),
    Document(page_content=DOC_LANGFUSE, metadata={"source": "langfuse_guide", "topic": "Langfuse"}),
]

total_chars = sum(len(doc.page_content) for doc in raw_docs)
print(f"총 문서 수: {len(raw_docs)}")
print(f"총 문자 수: {total_chars:,}")

---
## 섹션 4. 토큰 카운트 함수 (tiktoken)

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """텍스트의 토큰 수를 계산합니다."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

def count_docs_tokens(docs: list, model: str = "gpt-4o") -> int:
    """Document 리스트의 총 토큰 수를 계산합니다."""
    return sum(count_tokens(doc.page_content, model) for doc in docs)

def compression_ratio(original_tokens: int, compressed_tokens: int) -> float:
    """압축률을 계산합니다 (낮을수록 더 많이 압축)."""
    if original_tokens == 0:
        return 1.0
    return compressed_tokens / original_tokens

# 전체 문서의 토큰 수 측정
total_tokens = count_docs_tokens(raw_docs)
print(f"전체 문서 토큰 수: {total_tokens:,} tokens")
print(f"GPT-4o 입력 비용 (1회): ${total_tokens * 2.50 / 1_000_000:.4f}")
print(f"하루 1,000회 쿼리 시 월 비용: ${total_tokens * 2.50 / 1_000_000 * 1000 * 30:.2f}")

In [ ]:
# 문서를 청크로 분할하고 VectorStore 생성
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(raw_docs)
print(f"청크 수: {len(chunks)}")
print(f"평균 청크 크기: {sum(len(c.page_content) for c in chunks)//len(chunks)} 문자")

# VectorStore 생성 (Chroma)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ep07_docs"
)
print("VectorStore 생성 완료")

---
## 섹션 5. LLMChainFilter 압축 (LangChain 기본)

`LLMChainFilter`는 각 청크를 LLM이 검토하여 쿼리와 관련 없는 청크를 통째로 제거합니다.  
청크 단위로 포함/제외를 결정하므로 원문이 그대로 보존됩니다.

In [ ]:
# LLMChainFilter 설정
llm_filter = LLMChainFilter.from_llm(llm)

filter_retriever = ContextualCompressionRetriever(
    base_compressor=llm_filter,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)

# 테스트 쿼리
test_query = "What are the main context compression techniques in LangChain?"

base_docs = vectorstore.similarity_search(test_query, k=6)
filtered_docs = filter_retriever.invoke(test_query)

base_tokens = count_docs_tokens(base_docs)
filtered_tokens = count_docs_tokens(filtered_docs)

print(f"[LLMChainFilter]")
print(f"  압축 전: {len(base_docs)}개 청크, {base_tokens} tokens")
print(f"  압축 후: {len(filtered_docs)}개 청크, {filtered_tokens} tokens")
print(f"  압축률: {compression_ratio(base_tokens, filtered_tokens):.1%} (유지율)")
print(f"  토큰 절감: {base_tokens - filtered_tokens} tokens")

---
## 섹션 6. LLMChainExtractor 압축

`LLMChainExtractor`는 각 청크에서 쿼리와 관련된 문장만 추출합니다.  
LLM이 재작성하지 않고 원문에서 발췌하지만, 요약 형태가 될 수 있습니다.

In [ ]:
# LLMChainExtractor 설정
llm_extractor = LLMChainExtractor.from_llm(llm)

extractor_retriever = ContextualCompressionRetriever(
    base_compressor=llm_extractor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)

extracted_docs = extractor_retriever.invoke(test_query)
extracted_tokens = count_docs_tokens(extracted_docs)

print(f"[LLMChainExtractor]")
print(f"  압축 전: {len(base_docs)}개 청크, {base_tokens} tokens")
print(f"  압축 후: {len(extracted_docs)}개 청크, {extracted_tokens} tokens")
print(f"  압축률: {compression_ratio(base_tokens, extracted_tokens):.1%} (유지율)")
print(f"  토큰 절감: {base_tokens - extracted_tokens} tokens")
print()
print("--- 첫 번째 추출 결과 미리보기 ---")
if extracted_docs:
    print(extracted_docs[0].page_content[:300])

---
## 섹션 7. EmbeddingsFilter 압축 + Pipeline 구성

`EmbeddingsFilter`는 청크와 쿼리 간 임베딩 유사도가 임계값 미만이면 제거합니다.  
LLM 호출 없이 빠르게 동작합니다.

In [ ]:
# EmbeddingsFilter
emb_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.76
)

emb_retriever = ContextualCompressionRetriever(
    base_compressor=emb_filter,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)

emb_docs = emb_retriever.invoke(test_query)
emb_tokens = count_docs_tokens(emb_docs)

print(f"[EmbeddingsFilter (threshold=0.76)]")
print(f"  압축 전: {len(base_docs)}개 청크, {base_tokens} tokens")
print(f"  압축 후: {len(emb_docs)}개 청크, {emb_tokens} tokens")
print(f"  압축률: {compression_ratio(base_tokens, emb_tokens):.1%} (유지율)")

print()

# Pipeline: EmbeddingsFilter → LLMChainExtractor
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[emb_filter, llm_extractor]
)

pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)

pipeline_docs = pipeline_retriever.invoke(test_query)
pipeline_tokens = count_docs_tokens(pipeline_docs)

print(f"[Pipeline: EmbeddingsFilter → LLMChainExtractor]")
print(f"  압축 전: {len(base_docs)}개 청크, {base_tokens} tokens")
print(f"  압축 후: {len(pipeline_docs)}개 청크, {pipeline_tokens} tokens")
print(f"  압축률: {compression_ratio(base_tokens, pipeline_tokens):.1%} (유지율)")
print(f"  토큰 절감: {base_tokens - pipeline_tokens} tokens")

---
## 섹션 8. 압축률 측정 종합 비교

모든 압축기의 결과를 테이블로 정리합니다.

In [ ]:
import pandas as pd

results = [
    {"압축기": "압축 없음 (기본)", "청크 수": len(base_docs), "토큰 수": base_tokens, "유지율": 1.0},
    {"압축기": "EmbeddingsFilter", "청크 수": len(emb_docs), "토큰 수": emb_tokens,
     "유지율": compression_ratio(base_tokens, emb_tokens)},
    {"압축기": "LLMChainFilter", "청크 수": len(filtered_docs), "토큰 수": filtered_tokens,
     "유지율": compression_ratio(base_tokens, filtered_tokens)},
    {"압축기": "LLMChainExtractor", "청크 수": len(extracted_docs), "토큰 수": extracted_tokens,
     "유지율": compression_ratio(base_tokens, extracted_tokens)},
    {"압축기": "Pipeline (EmbF + Extractor)", "청크 수": len(pipeline_docs), "토큰 수": pipeline_tokens,
     "유지율": compression_ratio(base_tokens, pipeline_tokens)},
]

df = pd.DataFrame(results)
df["절감 토큰"] = base_tokens - df["토큰 수"]
df["유지율"] = df["유지율"].map("{:.1%}".format)
print(df.to_string(index=False))

---
## 섹션 9. 압축 전후 QA 정확도 비교

5개의 질문으로 각 압축기의 QA 품질을 측정합니다.

In [ ]:
from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

# QA 테스트 셋 (질문 + 키워드 정답)
qa_pairs = [
    {
        "question": "What is LLMlingua and how does it compress text?",
        "keywords": ["perplexity", "small language model", "phi", "token"]
    },
    {
        "question": "What are the main vector databases supported by LangChain?",
        "keywords": ["chroma", "pinecone", "faiss", "weaviate"]
    },
    {
        "question": "How does ContextualCompressionRetriever work?",
        "keywords": ["base retriever", "compressor", "compress"]
    },
    {
        "question": "What is Langfuse and what does it track?",
        "keywords": ["token", "cost", "trace", "observability"]
    },
    {
        "question": "What is the difference between RAG and long-context LLMs?",
        "keywords": ["context window", "cost", "knowledge base"]
    },
]

def evaluate_answer(answer: str, keywords: list) -> float:
    """키워드 포함 여부로 정확도를 측정합니다."""
    answer_lower = answer.lower()
    hits = sum(1 for kw in keywords if kw.lower() in answer_lower)
    return hits / len(keywords)

# RAG QA 프롬프트
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer based on the provided context only."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

def run_qa(retriever, question: str) -> str:
    """주어진 리트리버로 QA를 실행합니다."""
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    messages = qa_prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    return response.content

print("QA 평가 실행 중...")
retrievers = {
    "No Compression": vectorstore.as_retriever(search_kwargs={"k": 6}),
    "EmbeddingsFilter": emb_retriever,
    "LLMChainFilter": filter_retriever,
    "LLMChainExtractor": extractor_retriever,
    "Pipeline": pipeline_retriever,
}

accuracy_results = {}
for name, retriever in retrievers.items():
    scores = []
    for qa in qa_pairs:
        try:
            answer = run_qa(retriever, qa["question"])
            score = evaluate_answer(answer, qa["keywords"])
            scores.append(score)
        except Exception as e:
            scores.append(0.0)
    accuracy_results[name] = sum(scores) / len(scores)
    print(f"  {name}: {accuracy_results[name]:.1%}")

print("\nQA 평가 완료")

---
## 섹션 10. Langfuse CallbackHandler + 토큰 비용 추적

In [ ]:
from langfuse.langchain import CallbackHandler

# Langfuse 핸들러 초기화
langfuse_handler = CallbackHandler(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com",
    session_id="ep07-context-compression",
)

print("Langfuse 핸들러 초기화 완료")
print("대시보드: https://cloud.langfuse.com")

In [ ]:
# Langfuse로 추적하면서 Pipeline 압축 실행
tracked_query = "How does context compression reduce LLM costs?"

docs = pipeline_retriever.invoke(
    tracked_query,
    config={"callbacks": [langfuse_handler]}
)

context = "\n\n".join(doc.page_content for doc in docs)
messages = qa_prompt.format_messages(context=context, question=tracked_query)

response = llm.invoke(
    messages,
    config={"callbacks": [langfuse_handler]}
)

print(f"질문: {tracked_query}")
print(f"컨텍스트 토큰 수: {count_tokens(context)}")
print(f"응답: {response.content[:300]}...")
print("\nLangfuse 대시보드에서 토큰 사용량을 확인하세요!")

---
## 섹션 11. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- 그래프 1: 토큰 수 비교 막대그래프 ---
compressor_names = ["No\nCompress", "Emb\nFilter", "LLM\nFilter", "LLM\nExtractor", "Pipeline"]
token_values = [
    base_tokens,
    emb_tokens,
    filtered_tokens,
    extracted_tokens,
    pipeline_tokens
]

colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
bars = ax1.bar(compressor_names, token_values, color=colors, alpha=0.85, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, token_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_title("Token Count by Compressor", fontsize=13, fontweight='bold')
ax1.set_ylabel("Token Count")
ax1.set_ylim(0, max(token_values) * 1.2)
ax1.grid(axis='y', alpha=0.3)

# --- 그래프 2: 압축률 vs QA 정확도 scatter plot ---
retention_rates = [
    1.0,
    emb_tokens / base_tokens,
    filtered_tokens / base_tokens,
    extracted_tokens / base_tokens,
    pipeline_tokens / base_tokens,
]
acc_values = [
    accuracy_results.get("No Compression", 0.85),
    accuracy_results.get("EmbeddingsFilter", 0.80),
    accuracy_results.get("LLMChainFilter", 0.90),
    accuracy_results.get("LLMChainExtractor", 0.82),
    accuracy_results.get("Pipeline", 0.88),
]
labels = ["No Compress", "EmbFilter", "LLMFilter", "LLMExtractor", "Pipeline"]

for i, (x, y, label) in enumerate(zip(retention_rates, acc_values, labels)):
    ax2.scatter(x, y, s=120, color=colors[i], zorder=5)
    ax2.annotate(label, (x, y), textcoords="offset points", xytext=(8, 4), fontsize=9)

ax2.set_xlabel("Token Retention Rate (lower = more compressed)")
ax2.set_ylabel("QA Accuracy")
ax2.set_title("Compression Rate vs QA Accuracy", fontsize=13, fontweight='bold')
ax2.set_xlim(0, 1.15)
ax2.set_ylim(0.5, 1.05)
ax2.axhline(y=0.85, color='red', linestyle='--', alpha=0.4, label='85% accuracy threshold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("compression_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: compression_results.png")

---
## Exercise

### Exercise 1: 압축률 vs QA 정확도 실험

**목표**: `EmbeddingsFilter`의 `similarity_threshold`를 0.5 / 0.65 / 0.76 / 0.85로 바꿔가며 토큰 수와 QA 정확도 변화를 측정합니다.

아래 TODO를 완성하세요.

In [ ]:
thresholds = [0.50, 0.65, 0.76, 0.85]
exercise_results = []

for threshold in thresholds:
    # TODO 1: threshold 값으로 EmbeddingsFilter 생성
    # TODO 2: ContextualCompressionRetriever 생성
    # TODO 3: qa_pairs의 5개 질문으로 QA 실행 및 정확도 계산
    # TODO 4: 결과를 exercise_results 리스트에 추가
    pass

# TODO 5: 결과를 DataFrame으로 출력
# TODO 6: threshold vs 정확도 라인 차트 그리기 (x축: threshold, y축: 정확도 및 토큰 수 두 y축)

### Exercise 2: LLMlingua vs LangChain 압축기 비교

**목표**: `llmlingua` 라이브러리를 설치하고 LangChain 압축기와 결과를 비교합니다.

아래 TODO를 완성하세요.

In [ ]:
# !uv pip install llmlingua
# from llmlingua import PromptCompressor

# TODO 1: LLMlingua PromptCompressor 초기화 (model_name="microsoft/phi-2" 또는 "llmlingua-2")
# TODO 2: DOC_RAG 문서를 rate=0.5로 압축
# TODO 3: 압축 전후 토큰 수 측정 (tiktoken)
# TODO 4: 압축된 텍스트로 동일한 5개 QA 질문 실행
# TODO 5: LLMChainExtractor 결과와 정확도 비교 테이블 출력

# 힌트:
# llm_lingua = PromptCompressor(model_name="microsoft/phi-2", use_llmlingua2=False)
# compressed = llm_lingua.compress_prompt(context, rate=0.5, force_tokens=[".", "\n"])
# compressed_text = compressed["compressed_prompt"]

print("Exercise 2: TODO를 완성하세요!")